In [7]:
import asyncio
import json
import re
import sys
from typing import Any, Dict, List, Optional
from mcpuniverse.llm import AsyncVLLMEngine, TITOLLMWrapper
from mcpuniverse.mcp.manager import MCPManager
from mcpuniverse.agent.utils import get_tools_description


async def _init_mcp_servers(
    server_names: List[str],
    transport: str = "stdio",
) -> tuple:
    """
    Start MCP servers and return (mcp_manager, clients, tools_description).

    Args:
        server_names: e.g. ["weather", "calculator"]
        transport: "stdio" or "sse"

    Returns:
        (MCPManager, {name: MCPClient}, tools_description_str)
    """

    mcp_manager = MCPManager()
    clients = {}
    tools = {}

    for name in server_names:
        client = await mcp_manager.build_client(name, transport=transport)
        clients[name] = client
        tools[name] = await client.list_tools()
        print(f"    {name}: {len(tools[name])} tools loaded")

    tools_desc = get_tools_description(tools)
    return mcp_manager, clients, tools_desc


async def _call_mcp_tool(
    clients: Dict[str, Any],
    action: dict,
) -> str:
    """
    Execute a tool call via MCP.

    Args:
        clients: {server_name: MCPClient}
        action: {"server": "...", "tool": "...", "arguments": {...}}

    Returns:
        Tool result text.
    """
    server = action.get("server", "")
    tool = action.get("tool", "")
    arguments = action.get("arguments", {})

    if server not in clients:
        return f"Error: unknown server '{server}'. Available: {list(clients.keys())}"

    try:
        result_obj = await clients[server].execute_tool(tool, arguments)
        return result_obj.content[0].text
    except Exception as e:
        return f"Error calling {server}.{tool}: {e}"


async def _cleanup_mcp(clients: Dict[str, Any]):
    """Clean up all MCP clients."""
    for client in clients.values():
        try:
            await client.cleanup()
        except Exception:
            pass


# -- System prompt builder ---------------------------------------------------

def _build_system_prompt(tools_description: str) -> str:
    return (
        "You are a helpful assistant. You can use tools to help answer questions.\n\n"
        "At each step, respond with a single JSON object (no code fences):\n\n"
        "If you need to use a tool:\n"
        '{"thought": "your reasoning", '
        '"action": {"server": "server-name", "tool": "tool-name", '
        '"arguments": {"arg": "value"}}}\n\n'
        "If you have the final answer:\n"
        '{"thought": "your reasoning", "answer": "final answer"}\n\n'
        f"Available tools:\n{tools_description}"
    )


def _parse_json(text: str) -> dict:
    """Best-effort JSON extraction from LLM output."""
    clean = text.strip()
    for prefix in ("```json", "```"):
        if clean.startswith(prefix):
            clean = clean[len(prefix):]
    if clean.endswith("```"):
        clean = clean[:-3]
    clean = clean.strip()

    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
        return {"answer": text}

In [ ]:
async def run_batch_tito(
    model_path: str = "Qwen3-4B-Instruct-2507",
    instructions: Optional[List[str]] = None,
    mcp_servers: Optional[List[str]] = None,
    max_iterations: int = 5,
    **kwargs,
) -> List[Dict]:
    """
    Batch TITO processing with real MCP tools.

    Each instruction runs a full ReAct agent loop with MCP tool execution.
    Every instruction gets an independent TITOLLMWrapper, but all share
    the same AsyncVLLMEngine and MCP clients.
    """

    if instructions is None:
        instructions = [
            "What's the weather in San Francisco?",
            "What's the weather in Chicago?",
            "What's the weather in New York?",
        ]
    if mcp_servers is None:
        mcp_servers = ["weather"]

    print("=" * 70)
    print(f"Batch TITO: {len(instructions)} instructions, MCP servers: {mcp_servers}")
    print("=" * 70)

    # Shared engine
    engine = AsyncVLLMEngine(
        model_path=model_path,
        tensor_parallel_size=kwargs.get("tensor_parallel_size", 1),
        dtype="auto",
        trust_remote_code=True,
        max_model_len=4096,
        gpu_memory_utilization=kwargs.get("gpu_memory_utilization", 0.8),
    )
    await engine.init_engine()
    tokenizer = await engine.get_tokenizer()

    # Shared MCP clients
    _, clients, tools_desc = await _init_mcp_servers(mcp_servers)
    system_prompt = _build_system_prompt(tools_desc)

    results = []

    for i, instruction in enumerate(instructions):
        print(f"\n{'=' * 50}")
        print(f"Task {i + 1}/{len(instructions)}: {instruction}")
        print(f"{'=' * 50}")

        tito = TITOLLMWrapper(
            engine=engine,
            tokenizer=tokenizer,
            sampling_params={
                "temperature": kwargs.get("temperature", 0.7),
                "max_tokens": kwargs.get("max_tokens", 512),
                "stop": ["<|im_end|>"],
                "include_stop_str_in_output": False,
            },
            skip_special_tokens=False,
        )
        tito.reset_trajectory()

        prompt = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        # Agent loop with real MCP tool calling
        for iteration in range(max_iterations):
            response = await tito.generate_async(
                messages=[{"role": "raw", "content": prompt}],
            )
            parsed = _parse_json(response)

            if "answer" in parsed:
                print(f"  Iter {iteration + 1}: Final answer: {parsed['answer']}")
                break

            if "action" in parsed:
                action = parsed["action"]
                server = action.get("server", "")
                tool = action.get("tool", "")
                args = action.get("arguments", {})

                tool_result = await _call_mcp_tool(clients, action)
                print(f"  Iter {iteration + 1}: {server}.{tool}({args}) -> {tool_result[:200]}")

                tool_result_text = (
                    f"<|im_end|>\n"
                    f"<|im_start|>user\nTool execution result:\n{tool_result}<|im_end|>\n"
                    f"<|im_start|>assistant\n"
                )
                prompt = prompt + response + tool_result_text
            else:
                print(f"  Iter {iteration + 1}: Unexpected format, stopping")
                break

        trajectory = tito.get_trajectory()
        loss_mask = trajectory.get_loss_mask()

        print(f"  Tokens: {len(trajectory.token_ids)}, "
              f"Trainable: {sum(loss_mask)}, "
              f"Masked: {len(loss_mask) - sum(loss_mask)}, "
              f"Segments: {len(trajectory.segments)}")

        results.append({
            "instruction": instruction,
            "response": response,
            "token_ids": trajectory.token_ids,
            "loss_mask": loss_mask,
        })

    await _cleanup_mcp(clients)
    await engine.shutdown()

    print(f"\n{'=' * 70}")
    print("Batch complete")
    print(f"{'=' * 70}")

    return results

## TITO mode via RolloutEngine + Dispatcher

The `run_batch_tito` above is a standalone demo that manually handles the ReAct loop,
MCP tool calls, and token trajectory management. Below we show the **recommended**
approach: using `RolloutEngine` with `rollout_mode="token"` so that the full pipeline
(Dispatcher concurrency, Trajectory lifecycle, Evaluators) is available.

Key differences:
- `RolloutEngine` creates a shared `AsyncVLLMEngine` via `create_llm("async_vllm", ...)`
- Each trajectory gets its own `TITOLLMWrapper` automatically (created in `initialize_trajectory()`)
- The `ReActTrain` agent handles the ReAct loop and MCP tool calls
- `async_pipeline` dispatcher enables concurrent trajectory execution
- Evaluators compute rewards per trajectory
- Token-level data (`token_ids`, `trainable_mask`) is extracted automatically

In [ ]:
from mcpuniverse.rl import RolloutEngine, RolloutConfig


async def example_tito_with_rollout_engine():
    """
    TITO mode via RolloutEngine + Dispatcher.

    The engine automatically:
    1. Creates a shared AsyncVLLMEngine (token-level inference, no HTTP)
    2. Wraps it in per-trajectory TITOLLMWrapper during initialize_trajectory()
    3. Runs ReActTrain agents concurrently via async_pipeline dispatcher
    4. Extracts token_ids + trainable_mask from each trajectory
    """
    print("=" * 70)
    print("TITO Mode via RolloutEngine + Dispatcher")
    print("=" * 70)

    config = RolloutConfig.from_dict({
        "llm_type": "async_vllm",
        "rollout_mode": "token",
        "llm_config": {
            "model_path": "Qwen3-4B-Instruct-2507",
            "tensor_parallel_size": 1,
            "gpu_memory_utilization": 0.8,
            "max_model_len": 4096,
            "temperature": 0.7,
            "max_tokens": 512,
            # TITO-specific: stop token must be set explicitly
            # (text mode handles this at the HTTP API level)
            "stop": ["<|im_end|>"],
            "include_stop_str_in_output": False,
        },
        "agent_mode": "react_train",
        "formatter_type": "qwen3",
        "use_sample_servers": True,
        "generator": {
            "num_trajectories": 3,
            "max_iterations": 5,
        },
        "dispatcher": {
            "type": "async_pipeline",
            "max_parallel_agents": 8,
        },
    })

    engine = RolloutEngine(config)

    batch = [
        {
            "instruction": "What's the weather in San Francisco?",
            "instance_id": "sf_weather",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Francisco"},
            ],
        },
        {
            "instruction": "What's the weather in New York?",
            "instance_id": "nyc_weather",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "York"},
            ],
        },
        {
            "instruction": "What's the weather in Chicago?",
            "instance_id": "chicago_weather",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Chicago"},
            ],
        },
    ]

    output = await engine.run(batch)

    # ---- Results ----
    num_traj = config.generator.num_trajectories
    print(f"\nProcessed {len(batch)} instances, "
          f"{len(output.responses)} trajectories ({num_traj} per instance)")
    print(f"Success rate: "
          f"{output.rollout_metrics.get('rollout_metrics/success_rate', 0):.2%}")

    for traj in output.trajectories:
        iid = traj.get("instance_id", "?")
        tid = traj.get("trajectory_id", 0)
        token_ids = traj.get("token_ids", [])
        trainable_mask = traj.get("trainable_mask", [])
        reward = traj.get("reward", 0.0)
        steps = traj.get("num_steps", 0)
        tool_calls = traj.get("num_tool_calls", 0)

        print(f"\n  {iid}-{tid}:")
        print(f"    Reward: {reward}, Steps: {steps}, Tool calls: {tool_calls}")
        print(f"    Tokens: {len(token_ids)}, "
              f"Trainable: {sum(trainable_mask)}, "
              f"Masked: {len(trainable_mask) - sum(trainable_mask)}")

    return output